# GNN Model Training

Uses the tr_data, val_data, te_data and split indices produced by temporal_data_splitting.
Trains a GINe (Graph Isomorphism Network with Edge features) model to classify transactions as
legitimate (0) or money laundering (1).

1. Load Split Data Objects

In [1]:
import os
import torch

split_path = "split_data.pt" if os.path.exists("split_data.pt") else os.path.join("model", "split_data.pt")
split = torch.load(split_path, weights_only=False)

tr_data  = split["tr_data"]
val_data = split["val_data"]
te_data  = split["te_data"]
tr_inds  = split["tr_inds"]
val_inds = split["val_inds"]
te_inds  = split["te_inds"]

print("Train data :", tr_data)
print("Val data   :", val_data)
print("Test data  :", te_data)
print(f"Train indices : {tr_inds.shape[0]:,}")
print(f"Val indices   : {val_inds.shape[0]:,}")
print(f"Test indices  : {te_inds.shape[0]:,}")

Train data : Data(x=[2061626, 1], edge_index=[2, 10255698], edge_attr=[10255698, 4], y=[10255698], timestamps=[10255698])
Val data : Data(x=[2061626, 1], edge_index=[2, 13274841], edge_attr=[13274841, 4], y=[13274841], timestamps=[13274841])
Test data : Data(x=[2061626, 1], edge_index=[2, 15000000], edge_attr=[15000000, 4], y=[15000000], timestamps=[15000000])
Train indices : 10,255,698
Val indices : 3,019,143
Test indices : 1,725,159


2. Add Unique Edge IDs to Edge Attributes

Prepends a unique integer ID to the start of each edge's feature vector.
During mini-batch training, the loader samples subgraphs and shuffles edges.
This ID is the only way to track which edges in the batch correspond to
the original seed (training) edges we want to compute the loss on.
The ID is stripped off before feeding edge_attr into the model.

In [2]:
def add_arange_ids(data_list):
    for data in data_list:
        arange = torch.arange(data.edge_attr.shape[0]).view(-1, 1).float()
        data.edge_attr = torch.cat([arange, data.edge_attr], dim=1)

add_arange_ids([tr_data, val_data, te_data])

print(f'edge_attr shape after adding ID column: {tr_data.edge_attr.shape}')
print(f'Column 0 is the unique edge ID, columns 1-4 are the original features')
print(f'Sample row: {tr_data.edge_attr[0].tolist()}')

edge_attr shape after adding ID column: torch.Size([10255698, 5])
Column 0 is the unique edge ID, columns 1-4 are the original features
Sample row: [0.0, -1.241639494895935, -0.0019313407829031348, -0.5490716099739075, -1.1582497358322144]


3. Create Mini-Batch Graph DataLoaders

With 10M+ edges in the graph, the full graph cannot fit in GPU memory for a single forward pass.
LinkNeighborLoader creates mini-batches by:
1. Sampling a set of seed edges (transactions to classify)
2. For each seed edge's endpoint nodes, sampling their k-hop neighbors
3. Returning a small subgraph containing the seed edges + sampled neighborhood

num_neighbors=[100, 100] means: sample up to 100 neighbors at hop-1, then 100 neighbors at hop-2.

In [3]:
from torch_geometric.loader import LinkNeighborLoader

BATCH_SIZE = 8192
NUM_NEIGHBORS = [100, 100]

tr_loader = LinkNeighborLoader(
    tr_data,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = LinkNeighborLoader(
    val_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=val_data.edge_index[:, val_inds],
    edge_label=val_data.y[val_inds],
    batch_size=BATCH_SIZE,
    shuffle=False
)

te_loader = LinkNeighborLoader(
    te_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=te_data.edge_index[:, te_inds],
    edge_label=te_data.y[te_inds],
    batch_size=BATCH_SIZE,
    shuffle=False
)

sample_batch = next(iter(tr_loader))
print(f'Sample batch edge_attr shape : {sample_batch.edge_attr.shape}')
print(f'Sample batch nodes : {sample_batch.num_nodes}')
print(f'Sample batch edges : {sample_batch.num_edges}')

/home/shreyas-nalle/anaconda3/envs/mastermoney/lib/python3.9/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Sample batch edge_attr shape : torch.Size([264657, 5])
Sample batch nodes : 43359
Sample batch edges : 264657


4. Define the GINe Model

GINe (Graph Isomorphism Network with Edge features) is the model used from the repo.
Architecture:
- node_emb  : Linear layer projects node features from dim 1 -> n_hidden
- edge_emb  : Linear layer projects edge features from dim 4 -> n_hidden
- GINEConv layers: each layer aggregates neighbor node and edge information
  using a learnable MLP with residual connections and batch normalization
- Final MLP: takes [src_emb | dst_emb | edge_emb] (3 x n_hidden) and outputs 2 class logits

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm, Linear

class GINe(torch.nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes=2,
                 n_hidden=100, edge_updates=False,
                 edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates
        self.final_dropout = final_dropout

        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)

        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for _ in range(self.num_gnn_layers):
            conv = GINEConv(
                nn.Sequential(
                    nn.Linear(self.n_hidden, self.n_hidden),
                    nn.ReLU(),
                    nn.Linear(self.n_hidden, self.n_hidden)
                ),
                edge_dim=self.n_hidden
            )
            if self.edge_updates:
                self.emlps.append(nn.Sequential(
                    nn.Linear(3 * self.n_hidden, self.n_hidden),
                    nn.ReLU(),
                    nn.Linear(self.n_hidden, self.n_hidden),
                ))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))

        self.mlp = nn.Sequential(
            Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(self.final_dropout),
            Linear(50, 25),           nn.ReLU(), nn.Dropout(self.final_dropout),
            Linear(25, n_classes)
        )

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], dim=-1)) / 2

        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        x = torch.cat((x, edge_attr.view(-1, edge_attr.shape[1])), dim=1)
        return self.mlp(x)

N_HIDDEN = 64
N_GNN_LAYERS = 2
EDGE_DIM = sample_batch.edge_attr.shape[1] - 1 
NUM_FEATURES = sample_batch.x.shape[1]

model = GINe(
    num_features = NUM_FEATURES,
    num_gnn_layers = N_GNN_LAYERS,
    n_classes = 2,
    n_hidden = N_HIDDEN,
    edge_updates = True,
    edge_dim = EDGE_DIM,
    dropout = 0.0,
    final_dropout = 0.5
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model architecture:\n{model}')
print(f'Total trainable parameters: {total_params:,}')

Model architecture:
GINe(
  (node_emb): Linear(in_features=1, out_features=64, bias=True)
  (edge_emb): Linear(in_features=4, out_features=64, bias=True)
  (convs): ModuleList(
    (0-1): 2 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (emlps): ModuleList(
    (0-1): 2 x Sequential(
      (0): Linear(in_features=192, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (batch_norms): ModuleList(
    (0-1): 2 x BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (mlp): Sequential(
    (0): Linear(192, 50, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(50, 25, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(25, 2, bias=True)
  )
)
Total trainable parameters: 69,665


5. Loss Function, Optimizer and Device

Class Imbalance: only ~0.04% of transactions are money laundering.
Without handling this, the model will learn to predict everything as 0 (clean) and still get 99.96% accuracy!
Weighted Cross Entropy assigns a high penalty (w_ce2) to mislabelling the rare class (laundering=1),
and a low penalty (w_ce1) to mislabelling the majority class (clean=0).

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = model.to(device)

W_CE1 = 1.0  
W_CE2 = 50.0  
loss_fn = torch.nn.CrossEntropyLoss(
    weight=torch.FloatTensor([W_CE1, W_CE2]).to(device)
)

LR = 0.0003
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f'Loss : Weighted CrossEntropy (w_clean={W_CE1}, w_laundering={W_CE2})')
print(f'Optimizer: Adam (lr={LR})')

Training on: cpu
Loss : Weighted CrossEntropy (w_clean=1.0, w_laundering=50.0)
Optimizer: Adam (lr=0.0003)


6. Training Loop

For each epoch:
1. Loop over mini-batches from tr_loader
2. Use the edge ID (column 0) to identify the seed edges within the batch
3. Strip the edge ID before passing edge_attr to the model
4. Forward pass -> compute loss only on seed edges -> backpropagate
5. After each epoch: evaluate on val and test sets using F1 score

In [6]:
import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

@torch.no_grad()
def evaluate(loader, inds, model, data, device):
    model.eval()
    preds, ground_truths = [], []
    for batch in loader:
        inds_cpu = inds.detach().cpu()
        batch_edge_inds = inds_cpu[batch.input_id.detach().cpu()]
        batch_edge_ids  = loader.data.edge_attr.detach().cpu()[batch_edge_inds, 0]
        mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), batch_edge_ids)

        batch.edge_attr = batch.edge_attr[:, 1:]
        batch = batch.to(device)

        out  = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out[mask].argmax(dim=-1)
        preds.append(pred.cpu())
        ground_truths.append(batch.y[mask].cpu())

    pred = torch.cat(preds).numpy()
    ground_truth = torch.cat(ground_truths).numpy()
    return {
        'f1' : f1_score(ground_truth, pred, zero_division=0),
        'precision': precision_score(ground_truth, pred, zero_division=0),
        'recall' : recall_score(ground_truth, pred, zero_division=0)
    }

N_EPOCHS = 10
best_val_f1 = 0.0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = total_examples = 0
    preds, ground_truths = [], []

    for batch in tqdm.tqdm(tr_loader, desc=f'Epoch {epoch}/{N_EPOCHS}'):
        optimizer.zero_grad()

        inds_cpu = tr_inds.detach().cpu()
        batch_edge_inds = inds_cpu[batch.input_id.detach().cpu()]
        batch_edge_ids = tr_loader.data.edge_attr.detach().cpu()[batch_edge_inds, 0]
        mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), batch_edge_ids)

        batch.edge_attr = batch.edge_attr[:, 1:]
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out[mask]
        ground_truth = batch.y[mask]

        preds.append(pred.argmax(dim=-1).detach().cpu())
        ground_truths.append(ground_truth.detach().cpu())

        loss = loss_fn(pred, ground_truth)
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * pred.numel()
        total_examples += pred.numel()

    train_pred = torch.cat(preds).numpy()
    train_gt = torch.cat(ground_truths).numpy()
    train_f1 = f1_score(train_gt, train_pred, zero_division=0)

    val_metrics = evaluate(val_loader, val_inds, model, val_data, device)
    te_metrics = evaluate(te_loader,  te_inds,  model, te_data,  device)

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch {epoch:02d} | Loss: {total_loss/total_examples:.4f} | '
          f'Train F1: {train_f1:.4f} | '
          f'Val F1: {val_metrics["f1"]:.4f} | '
          f'Test F1: {te_metrics["f1"]:.4f} | '
          f'Test Precision: {te_metrics["precision"]:.4f} | '
          f'Test Recall: {te_metrics["recall"]:.4f}')

Epoch 1/10: 100%|██████████| 1252/1252 [30:39<00:00,  1.47s/it]


Epoch 01 | Loss: 0.0877 | Train F1: 0.0023 | Val F1: 0.0047 | Test F1: 0.0012 | Test Precision: 0.0638 | Test Recall: 0.0006


Epoch 2/10:  52%|█████▏    | 649/1252 [19:28<18:05,  1.80s/it]


KeyboardInterrupt: 

7. Final Evaluation on Test Set with Best Model

In [ ]:
# Load the best checkpoint saved during training
model.load_state_dict(torch.load('best_model.pt', map_location=device))

final_metrics = evaluate(te_loader, te_inds, model, te_data, device)

print('Final Test Set Results (Best Val Checkpoint):')
print(f'  F1 Score  : {final_metrics["f1"]:.4f}')
print(f'  Precision : {final_metrics["precision"]:.4f}')
print(f'  Recall    : {final_metrics["recall"]:.4f}')